# parents-dict-by-argidx — ex3: dispatch back fns over parents.items() — int-keyed positional only

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `parents-dict-by-argidx`. Running the final beacon cell reports progress against the `Backprop: Parents dict by argidx` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parents dict by argidx` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parents-dict-by-argidx`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parents-dict-by-argidx"
DD_SUBTOPIC = "Backprop: Parents dict by argidx"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Iterating `parents.items()` to dispatch back fns by argnum

Ex1 built `{argnum: tensor}` for positional Tensors; ex2 extended with kwarg-name keys. The deepening move USES the dict at reverse time: iterate `parents.items()`, look up `BACK_FUNCS[(fn, argnum)]`, compute that parent's grad contribution.

```python
for argnum, parent in recipe.parents.items():
    if not isinstance(argnum, int):
        continue  # kwarg parents handled separately
    back_fn = BACK_FUNCS[(recipe.func, argnum)]
    grad_for_parent = back_fn(grad_out, out_arr, *recipe.args)
    accumulate(parent, grad_for_parent)
```

**Why int-only filtering.** ex2 mixed int keys (positional) and str keys (kwarg names). The dispatcher for POSITIONAL back fns takes the int keys. Kwarg-positioned Tensors get a SEPARATE back-fn registry keyed by `(fn, kwarg_name: str)` — different table, same call shape.

**`*recipe.args` un-splats stored positional args.** The back fn signature is `(grad_out, out, *forward_args)`. Storing args as a tuple and splatting at call time is the cleanest way to feed the back fn whatever the forward had — `add` has 2 args, `clamp` has 1 (plus min/max kwargs), `sum` has 1 (plus dim kwarg). Splat handles all of them uniformly.

### Exercise 3 — dispatch back fns over parents.items() — int-keyed positional only

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the parents-dict dispatch pattern: iterate `recipe.parents.items()` filtering to int keys, look up each back fn by `(recipe.func, argnum)`, and return a mapping from argnum to the computed grad tensor.
> Keywords: parents, dispatch, argidx, iter-items
> ```

**KCs targeted:** `parents-dict-by-argidx`, `back-fn-lookup-by-fn-argnum`

Implement `ex3_dispatch_back_fns(recipe, grad_out, out_arr, back_funcs)`. Iterate over `recipe.parents.items()`, skip non-int keys (they're kwarg-name keys for kwarg-positioned Tensors), and for each `(argnum, parent_tensor)`:

1. Look up the back fn: `back_fn = back_funcs[(recipe.func, argnum)]`. If missing, raise `KeyError` (let the natural error propagate).
2. Call `back_fn(grad_out, out_arr, *recipe.args, **recipe.kwargs)` — splat both stored args and kwargs.
3. Collect the result keyed by `argnum`.

Return a `dict[int, Tensor]` mapping argnum → computed grad.

Inputs you can rely on:
- `recipe.func`: the forward fn (e.g. `t.add`).
- `recipe.args`: tuple of raw (unboxed) args used in the forward.
- `recipe.kwargs`: dict (possibly empty).
- `recipe.parents`: dict mixing int and str keys.
- `back_funcs`: dict keyed by `(fwd_fn, argnum: int)`.

Constraints:
- DO NOT compute grads for str-keyed parents (kwarg Tensors). Filter them out via `isinstance(k, int)`.
- DO NOT mutate `recipe`.

In [ ]:
def ex3_dispatch_back_fns(recipe, grad_out, out_arr, back_funcs):
    grads = {}
    for argnum, parent in recipe.parents.items():
        if not isinstance(argnum, int):
            continue  # kwarg-named parents handled by a different dispatcher
        back_fn = back_funcs[(recipe.func, argnum)]
        grads[argnum] = back_fn(
            grad_out, out_arr, *recipe.args, **recipe.kwargs
        )
    return grads


<details><summary>Solution</summary>

```python
def ex3_dispatch_back_fns(recipe, grad_out, out_arr, back_funcs):
    grads = {}
    for argnum, parent in recipe.parents.items():
        if not isinstance(argnum, int):
            continue  # kwarg-named parents handled by a different dispatcher
        back_fn = back_funcs[(recipe.func, argnum)]
        grads[argnum] = back_fn(
            grad_out, out_arr, *recipe.args, **recipe.kwargs
        )
    return grads
```

**`isinstance(argnum, int)` filter is critical.** ex2 mixed int and str keys in parents. The positional back-fn dispatcher must skip the str keys — they belong to a parallel kwarg-back-fn table that this drill doesn't model. Without the filter, you'd do `back_funcs[(recipe.func, 'mask')]` which crashes.

**`*recipe.args, **recipe.kwargs` is the universal call shape.** Every back fn's signature is `(grad_out, out, *forward_args, **forward_kwargs)`. Splatting works for unary, binary, ternary, and parameterized ops uniformly — that's the whole point of the Recipe abstraction.

**The dispatcher is intentionally minimal.** No grad accumulation (that's the leaf-accumulator's job); no graph traversal (that's topological sort's job). Just: 'given a Recipe and an incoming grad, hand back the per-argnum grad contributions'. Single responsibility.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()